
# Business Decision and Risk Analysis


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Define project directories
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

# Load recommendations generated in NB4
recommendations = pd.read_csv(
    PROCESSED_DIR / "inventory_recommendations.csv"
)

# Load simulated approval history
approval_log = pd.read_csv(
    PROCESSED_DIR / "approval_log_demo.csv"
)

# Verify loaded data
print("Recommendations:", recommendations.shape)
print("Approval records:", approval_log.shape)

print(
    "Missing recommendation values:",
    recommendations.isna().sum().sum()
)

print(
    "Pending recommendations:",
    (recommendations["approval_status"] == "PENDING").sum()
)

display(
    recommendations[
        ["item_id", "priority",
         "recommended_order_qty", "approval_status"]
    ].head()
)


Recommendations: (273, 20)
Approval records: (1, 6)
Missing recommendation values: 0
Pending recommendations: 273


,item_id,priority,recommended_order_qty,approval_status
0,FOODS_3_090,HIGH,744,PENDING
1,FOODS_3_252,HIGH,735,PENDING
2,FOODS_3_120,HIGH,632,PENDING
3,FOODS_3_586,HIGH,470,PENDING
4,FOODS_3_587,HIGH,402,PENDING



### 2. Generate Simulated Supplier Unit Costs


In [2]:

# Use a fixed seed so results can be reproduced
rng = np.random.default_rng(42)

# Generate simulated supplier costs in USD
recommendations["supplier_unit_cost"] = np.round(
    rng.uniform(
        low=1.0,
        high=25.0,
        size=len(recommendations)
    ),
    2
)

# Verify results
print("Total recommendations:", len(recommendations))

print(
    "Minimum supplier cost:",
    recommendations["supplier_unit_cost"].min()
)

print(
    "Maximum supplier cost:",
    recommendations["supplier_unit_cost"].max()
)

print(
    "Missing costs:",
    recommendations["supplier_unit_cost"].isna().sum()
)

print(
    "Invalid costs:",
    (recommendations["supplier_unit_cost"] <= 0).sum()
)

display(
    recommendations[
        ["item_id", "recommended_order_qty",
         "supplier_unit_cost"]
    ].head()
)


Total recommendations: 273
Minimum supplier cost: 1.18
Maximum supplier cost: 24.82
Missing costs: 0
Invalid costs: 0


,item_id,recommended_order_qty,supplier_unit_cost
0,FOODS_3_090,744,19.57
1,FOODS_3_252,735,11.53
2,FOODS_3_120,632,21.61
3,FOODS_3_586,470,17.74
4,FOODS_3_587,402,3.26



### 3. Calculate Estimated Purchasing Costs


In [3]:

# Calculate purchasing cost for each product
recommendations["estimated_order_cost"] = (
    recommendations["recommended_order_qty"]
    * recommendations["supplier_unit_cost"]
).round(2)

# Calculate total purchasing cost
total_purchasing_cost = (
    recommendations["estimated_order_cost"].sum()
)

# Verify calculations
print("Total products:", len(recommendations))

print(
    "Total recommended units:",
    recommendations["recommended_order_qty"].sum()
)

print(
    "Total estimated purchasing cost:",
    f"${total_purchasing_cost:,.2f}"
)

print(
    "Missing costs:",
    recommendations["estimated_order_cost"].isna().sum()
)

print(
    "Negative costs:",
    (recommendations["estimated_order_cost"] < 0).sum()
)

display(
    recommendations[
        [
            "item_id",
            "recommended_order_qty",
            "supplier_unit_cost",
            "estimated_order_cost"
        ]
    ].head()
)


Total products: 273
Total recommended units: 19045
Total estimated purchasing cost: $258,329.08
Missing costs: 0
Negative costs: 0


,item_id,recommended_order_qty,supplier_unit_cost,estimated_order_cost
0,FOODS_3_090,744,19.57,14560.08
1,FOODS_3_252,735,11.53,8474.55
2,FOODS_3_120,632,21.61,13657.52
3,FOODS_3_586,470,17.74,8337.80
4,FOODS_3_587,402,3.26,1310.52



### 4. Define Purchasing Budget and Funding Gap


In [4]:

# Set a simulated purchasing budget
purchasing_budget = 100_000.00

# Calculate the difference between budget and total cost
funding_gap = max(
    total_purchasing_cost - purchasing_budget,
    0
)

# Calculate the percentage of costs the budget can cover
budget_coverage = (
    purchasing_budget / total_purchasing_cost
) * 100

# Check whether the business can afford everything
can_afford_all = (
    purchasing_budget >= total_purchasing_cost
)

print(f"Available budget: ${purchasing_budget:,.2f}")
print(f"Total purchasing cost: ${total_purchasing_cost:,.2f}")
print(f"Funding gap: ${funding_gap:,.2f}")
print(f"Budget coverage: {budget_coverage:.2f}%")
print("Can afford all recommendations:", can_afford_all)


Available budget: $100,000.00
Total purchasing cost: $258,329.08
Funding gap: $158,329.08
Budget coverage: 38.71%
Can afford all recommendations: False



### 5. Identify Potential Stockout Risk


In [5]:

# Estimate days of stock remaining
recommendations["days_of_stock"] = np.where(
    recommendations["avg_daily_demand"] > 0,
    recommendations["current_stock"]
    / recommendations["avg_daily_demand"].clip(lower=1e-9),
    np.inf
)

# Flag products that may run out before delivery
recommendations["stockout_risk"] = np.where(
    recommendations["days_of_stock"]
    < recommendations["lead_time_days"],
    "AT_RISK",
    "LOW_RISK"
)

# Verify results
print("Total recommendations:", len(recommendations))

print("\nStockout risk distribution:")
print(recommendations["stockout_risk"].value_counts())

print(
    "\nMissing risk values:",
    recommendations["stockout_risk"].isna().sum()
)

# Display products with the shortest stock coverage
display(
    recommendations[
        [
            "item_id",
            "priority",
            "current_stock",
            "avg_daily_demand",
            "lead_time_days",
            "days_of_stock",
            "stockout_risk"
        ]
    ].sort_values(
        "days_of_stock"
    ).head(10).round(2)
)


Total recommendations: 273

Stockout risk distribution:
stockout_risk
AT_RISK     173
LOW_RISK    100
Name: count, dtype: int64

Missing risk values: 0


,item_id,priority,current_stock,avg_daily_demand,lead_time_days,days_of_stock,stockout_risk
0,FOODS_3_090,HIGH,30,65.77,5,0.46,AT_RISK
14,FOODS_3_555,HIGH,14,18.33,6,0.76,AT_RISK
2,FOODS_3_120,HIGH,63,54.85,6,1.15,AT_RISK
22,HOUSEHOLD_1_514,HIGH,10,8.49,11,1.18,AT_RISK
8,FOODS_3_511,HIGH,22,18.03,9,1.22,AT_RISK
4,FOODS_3_587,HIGH,36,27.98,9,1.29,AT_RISK
3,FOODS_3_586,HIGH,56,41.45,6,1.35,AT_RISK
1,FOODS_3_252,HIGH,57,37.54,14,1.52,AT_RISK
75,FOODS_1_218,MEDIUM,19,12.33,4,1.54,AT_RISK
26,HOUSEHOLD_1_465,HIGH,15,9.62,8,1.56,AT_RISK



### 6. Estimate Potential Shortages Before Supplier Delivery


In [6]:

# Estimate how many units customers may need
# before a new supplier order arrives
recommendations["demand_before_delivery"] = (
    recommendations["avg_daily_demand"]
    * recommendations["lead_time_days"]
)

# Estimate potential shortage using current stock
recommendations["potential_shortage_units"] = np.maximum(
    recommendations["demand_before_delivery"]
    - recommendations["current_stock"],
    0
)

# Display results
print(
    "Products with potential shortages:",
    (recommendations["potential_shortage_units"] > 0).sum()
)

print(
    "Total estimated shortage units:",
    round(recommendations["potential_shortage_units"].sum(), 2)
)

print(
    "Missing values:",
    recommendations["potential_shortage_units"].isna().sum()
)

display(
    recommendations[
        [
            "item_id",
            "priority",
            "current_stock",
            "lead_time_days",
            "demand_before_delivery",
            "potential_shortage_units"
        ]
    ].sort_values(
        "potential_shortage_units",
        ascending=False
    ).head(10).round(2)
)


Products with potential shortages: 173
Total estimated shortage units: 7286.45
Missing values: 0


,item_id,priority,current_stock,lead_time_days,demand_before_delivery,potential_shortage_units
1,FOODS_3_252,HIGH,57,14,525.63,468.63
0,FOODS_3_090,HIGH,30,5,328.84,298.84
2,FOODS_3_120,HIGH,63,6,329.12,266.12
4,FOODS_3_587,HIGH,36,9,251.81,215.81
3,FOODS_3_586,HIGH,56,6,248.69,192.69
6,FOODS_2_019,HIGH,46,14,214.58,168.58
7,FOODS_3_723,HIGH,63,14,222.59,159.59
11,FOODS_3_491,HIGH,52,14,198.67,146.67
10,FOODS_2_197,HIGH,64,14,208.45,144.45
9,FOODS_3_295,HIGH,100,14,241.48,141.48



### 7. Estimate Potential Lost Gross Profit from Stockouts


In [7]:

# Assume supplier cost represents 70% of selling price
gross_margin_rate = 0.30

# Estimate selling price from simulated supplier cost
recommendations["estimated_selling_price"] = (
    recommendations["supplier_unit_cost"]
    / (1 - gross_margin_rate)
)

# Estimate gross profit earned per unit sold
recommendations["gross_profit_per_unit"] = (
    recommendations["estimated_selling_price"]
    - recommendations["supplier_unit_cost"]
)

# Calculate potential lost gross profit
recommendations["potential_lost_profit"] = (
    recommendations["potential_shortage_units"]
    * recommendations["gross_profit_per_unit"]
)

# Verify results
print(
    "Products with potential lost profit:",
    (recommendations["potential_lost_profit"] > 0).sum()
)

print(
    "Total potential lost gross profit:",
    f"${recommendations['potential_lost_profit'].sum():,.2f}"
)

print(
    "Missing values:",
    recommendations["potential_lost_profit"].isna().sum()
)

display(
    recommendations[
        [
            "item_id",
            "potential_shortage_units",
            "gross_profit_per_unit",
            "potential_lost_profit"
        ]
    ].sort_values(
        "potential_lost_profit",
        ascending=False
    ).head(10).round(2)
)


Products with potential lost profit: 173
Total potential lost gross profit: $43,267.50
Missing values: 0


,item_id,potential_shortage_units,gross_profit_per_unit,potential_lost_profit
0,FOODS_3_090,298.84,8.39,2506.44
2,FOODS_3_120,266.12,9.26,2464.61
1,FOODS_3_252,468.63,4.94,2315.70
3,FOODS_3_586,192.69,7.60,1465.02
11,FOODS_3_491,146.67,9.96,1460.85
5,FOODS_3_064,133.38,10.46,1395.35
6,FOODS_2_019,168.58,8.26,1392.19
7,FOODS_3_723,159.59,8.52,1358.99
13,FOODS_3_329,116.08,8.89,1032.28
18,FOODS_3_281,108.34,8.94,968.59



### 8. Rank Products by Potential Financial Risk



In [8]:

# Rank products by estimated potential lost profit
financial_risk = recommendations.sort_values(
    "potential_lost_profit",
    ascending=False
).copy()

# Assign financial risk ranks
financial_risk["financial_risk_rank"] = np.arange(
    1, len(financial_risk) + 1
)

# Calculate the percentage of total financial exposure
total_risk = financial_risk["potential_lost_profit"].sum()

financial_risk["risk_share_percent"] = (
    financial_risk["potential_lost_profit"]
    / total_risk
    * 100
)

# Display the 10 highest-risk products
display(
    financial_risk[
        [
            "financial_risk_rank",
            "item_id",
            "priority",
            "stockout_risk",
            "potential_shortage_units",
            "potential_lost_profit",
            "risk_share_percent"
        ]
    ].head(10).round(2)
)

print("Total ranked products:", len(financial_risk))

print(
    "Top 10 potential lost profit:",
    f"${financial_risk['potential_lost_profit'].head(10).sum():,.2f}"
)

print(
    "Top 10 share of financial risk:",
    f"{financial_risk['risk_share_percent'].head(10).sum():.2f}%"
)


,financial_risk_rank,item_id,priority,stockout_risk,potential_shortage_units,potential_lost_profit,risk_share_percent
0,1,FOODS_3_090,HIGH,AT_RISK,298.84,2506.44,5.79
2,2,FOODS_3_120,HIGH,AT_RISK,266.12,2464.61,5.70
1,3,FOODS_3_252,HIGH,AT_RISK,468.63,2315.70,5.35
3,4,FOODS_3_586,HIGH,AT_RISK,192.69,1465.02,3.39
11,5,FOODS_3_491,HIGH,AT_RISK,146.67,1460.85,3.38
5,6,FOODS_3_064,HIGH,AT_RISK,133.38,1395.35,3.22
6,7,FOODS_2_019,HIGH,AT_RISK,168.58,1392.19,3.22
7,8,FOODS_3_723,HIGH,AT_RISK,159.59,1358.99,3.14
13,9,FOODS_3_329,HIGH,AT_RISK,116.08,1032.28,2.39
18,10,FOODS_3_281,HIGH,AT_RISK,108.34,968.59,2.24


Total ranked products: 273
Top 10 potential lost profit: $16,360.03
Top 10 share of financial risk: 37.81%



### 9. Allocate Purchasing Budget by Financial Risk


In [9]:

# Sort by estimated financial exposure
budget_plan = recommendations.sort_values(
    "potential_lost_profit",
    ascending=False
).copy()

# Track how much money is still available
remaining_budget = purchasing_budget

# Store approved-for-planning quantities
allocated_quantities = []

for _, row in budget_plan.iterrows():

    unit_cost = row["supplier_unit_cost"]
    requested_qty = int(row["recommended_order_qty"])

    # Maximum number of units we can afford
    affordable_qty = int(
        (remaining_budget + 1e-9) // unit_cost
    )

    # Never allocate more than requested
    allocated_qty = min(
        requested_qty,
        affordable_qty
    )

    allocated_quantities.append(allocated_qty)

    # Deduct estimated cost from budget
    remaining_budget = round(
        remaining_budget - allocated_qty * unit_cost,
        2
    )

# Save proposed quantities
budget_plan["allocated_order_qty"] = allocated_quantities

budget_plan["allocated_order_cost"] = (
    budget_plan["allocated_order_qty"]
    * budget_plan["supplier_unit_cost"]
).round(2)

# Verify financial constraints
print(
    "Total allocated units:",
    budget_plan["allocated_order_qty"].sum()
)

print(
    "Total allocated cost:",
    f"${budget_plan['allocated_order_cost'].sum():,.2f}"
)

print(
    "Remaining budget:",
    f"${remaining_budget:,.2f}"
)

print(
    "Products receiving allocations:",
    (budget_plan["allocated_order_qty"] > 0).sum()
)

print(
    "Budget exceeded:",
    budget_plan["allocated_order_cost"].sum()
    > purchasing_budget + 0.01
)


Total allocated units: 5370
Total allocated cost: $99,999.72
Remaining budget: $0.28
Products receiving allocations: 19
Budget exceeded: False



### 10. Evaluate Budget Allocation Coverage


In [10]:

# Classify each proposed allocation
budget_plan["allocation_status"] = np.select(
    [
        budget_plan["allocated_order_qty"]
        == budget_plan["recommended_order_qty"],

        budget_plan["allocated_order_qty"] > 0
    ],
    [
        "FULL",
        "PARTIAL"
    ],
    default="NOT_FUNDED"
)

# Calculate quantities not covered by our budget
budget_plan["unfunded_units"] = (
    budget_plan["recommended_order_qty"]
    - budget_plan["allocated_order_qty"]
)

# Display results
print("Allocation status:")
print(budget_plan["allocation_status"].value_counts())

print(
    "\nTotal unfunded units:",
    budget_plan["unfunded_units"].sum()
)

print(
    "AT_RISK products without funding:",
    (
        (budget_plan["stockout_risk"] == "AT_RISK")
        & (budget_plan["allocated_order_qty"] == 0)
    ).sum()
)

print(
    "Negative unfunded quantities:",
    (budget_plan["unfunded_units"] < 0).sum()
)


Allocation status:
allocation_status
NOT_FUNDED    254
FULL           16
PARTIAL         3
Name: count, dtype: int64

Total unfunded units: 13675
AT_RISK products without funding: 154
Negative unfunded quantities: 0



### 11. Identify Unfunded High-Risk Products


In [11]:

# Find at-risk products that received no budget
unfunded_high_risk = budget_plan.loc[
    (budget_plan["stockout_risk"] == "AT_RISK")
    & (budget_plan["allocation_status"] == "NOT_FUNDED")
].copy()

# Sort by potential lost gross profit
unfunded_high_risk = unfunded_high_risk.sort_values(
    "potential_lost_profit",
    ascending=False
)

# Summarize financial exposure
print(
    "Unfunded at-risk products:",
    len(unfunded_high_risk)
)

print(
    "Unfunded at-risk units requested:",
    unfunded_high_risk["unfunded_units"].sum()
)

print(
    "Potential lost gross profit exposure:",
    f"${unfunded_high_risk['potential_lost_profit'].sum():,.2f}"
)

print(
    "Missing values:",
    unfunded_high_risk[
        ["unfunded_units", "potential_lost_profit"]
    ].isna().sum().sum()
)

display(
    unfunded_high_risk[
        [
            "item_id",
            "priority",
            "unfunded_units",
            "lead_time_days",
            "potential_lost_profit"
        ]
    ].head(10).round(2)
)


Unfunded at-risk products: 154
Unfunded at-risk units requested: 11202
Potential lost gross profit exposure: $21,043.41
Missing values: 0


,item_id,priority,unfunded_units,lead_time_days,potential_lost_profit
10,FOODS_2_197,HIGH,258,14,612.89
16,FOODS_3_739,HIGH,213,9,612.25
72,HOUSEHOLD_1_418,MEDIUM,132,12,584.04
23,HOUSEHOLD_1_096,HIGH,145,13,560.72
29,FOODS_3_516,HIGH,131,11,556.27
63,FOODS_3_282,MEDIUM,224,5,513.52
14,FOODS_3_555,HIGH,230,6,478.93
66,FOODS_3_741,MEDIUM,169,11,471.68
30,FOODS_3_455,HIGH,126,11,462.14
64,FOODS_3_681,MEDIUM,200,8,453.49



### 12. Calculate Stock Shortage Severity


In [12]:

# Calculate how many days stock may run out
# before the supplier's delivery
budget_plan["shortage_gap_days"] = np.maximum(
    budget_plan["lead_time_days"]
    - budget_plan["days_of_stock"],
    0
)

# Identify the severity using simple rules
budget_plan["shortage_severity"] = np.select(
    [
        budget_plan["shortage_gap_days"] >= 7,
        budget_plan["shortage_gap_days"] > 0
    ],
    [
        "SEVERE",
        "MODERATE"
    ],
    default="NONE"
)

# Verify results
print("Shortage severity distribution:")
print(budget_plan["shortage_severity"].value_counts())

print(
    "\nSevere products without funding:",
    (
        (budget_plan["shortage_severity"] == "SEVERE")
        & (budget_plan["allocated_order_qty"] == 0)
    ).sum()
)

print(
    "\nMissing values:",
    budget_plan["shortage_severity"].isna().sum()
)

display(
    budget_plan[
        [
            "item_id",
            "days_of_stock",
            "lead_time_days",
            "shortage_gap_days",
            "shortage_severity",
            "allocation_status"
        ]
    ].sort_values(
        "shortage_gap_days",
        ascending=False
    ).head(10).round(2)
)


Shortage severity distribution:
shortage_severity
MODERATE    135
NONE        100
SEVERE       38
Name: count, dtype: int64

Severe products without funding: 23

Missing values: 0


,item_id,days_of_stock,lead_time_days,shortage_gap_days,shortage_severity,allocation_status
1,FOODS_3_252,1.52,14,12.48,SEVERE,FULL
35,FOODS_3_274,2.66,14,11.34,SEVERE,NOT_FUNDED
6,FOODS_2_019,3.00,14,11.00,SEVERE,FULL
11,FOODS_3_491,3.66,14,10.34,SEVERE,FULL
19,FOODS_1_200,1.68,12,10.32,SEVERE,FULL
25,FOODS_3_219,3.72,14,10.28,SEVERE,NOT_FUNDED
31,FOODS_3_228,2.82,13,10.18,SEVERE,PARTIAL
7,FOODS_3_723,3.96,14,10.04,SEVERE,FULL
28,HOUSEHOLD_1_339,2.07,12,9.93,SEVERE,NOT_FUNDED
39,FOODS_3_734,4.17,14,9.83,SEVERE,NOT_FUNDED



### 13. Identify Urgent Supplier Intervention Needs


In [13]:

# Identify actions based on estimated shortage severity
budget_plan["supplier_action"] = np.select(
    [
        budget_plan["shortage_severity"] == "SEVERE",
        budget_plan["shortage_severity"] == "MODERATE"
    ],
    [
        "REQUEST_EXPEDITED_DELIVERY",
        "REVIEW_DELIVERY_SCHEDULE"
    ],
    default="STANDARD_MONITORING"
)

# Verify actions
print("Supplier action distribution:")
print(budget_plan["supplier_action"].value_counts())

print(
    "\nUnfunded products needing expedited delivery:",
    (
        (budget_plan["supplier_action"] == "REQUEST_EXPEDITED_DELIVERY")
        & (budget_plan["allocated_order_qty"] == 0)
    ).sum()
)

print(
    "\nMissing values:",
    budget_plan["supplier_action"].isna().sum()
)

display(
    budget_plan[
        [
            "item_id",
            "shortage_severity",
            "allocation_status",
            "supplier_action"
        ]
    ].head(10)
)


Supplier action distribution:
supplier_action
REVIEW_DELIVERY_SCHEDULE      135
STANDARD_MONITORING           100
REQUEST_EXPEDITED_DELIVERY     38
Name: count, dtype: int64

Unfunded products needing expedited delivery: 23

Missing values: 0


,item_id,shortage_severity,allocation_status,supplier_action
0,FOODS_3_090,MODERATE,FULL,REVIEW_DELIVERY_SCHEDULE
2,FOODS_3_120,MODERATE,FULL,REVIEW_DELIVERY_SCHEDULE
1,FOODS_3_252,SEVERE,FULL,REQUEST_EXPEDITED_DELIVERY
3,FOODS_3_586,MODERATE,FULL,REVIEW_DELIVERY_SCHEDULE
11,FOODS_3_491,SEVERE,FULL,REQUEST_EXPEDITED_DELIVERY
5,FOODS_3_064,MODERATE,FULL,REVIEW_DELIVERY_SCHEDULE
6,FOODS_2_019,SEVERE,FULL,REQUEST_EXPEDITED_DELIVERY
7,FOODS_3_723,SEVERE,FULL,REQUEST_EXPEDITED_DELIVERY
13,FOODS_3_329,SEVERE,FULL,REQUEST_EXPEDITED_DELIVERY
18,FOODS_3_281,SEVERE,FULL,REQUEST_EXPEDITED_DELIVERY



### 14. Calculate Financial Exposure per Purchasing Dollar


In [14]:

# Compare financial exposure with proposed purchasing cost
budget_plan["exposure_per_dollar"] = (
    budget_plan["potential_lost_profit"]
    / budget_plan["estimated_order_cost"]
)

# Rank products by this ratio
exposure_ranking = budget_plan.sort_values(
    "exposure_per_dollar",
    ascending=False
).copy()

# Verify results
print("Total products:", len(exposure_ranking))

print(
    "Missing exposure ratios:",
    exposure_ranking["exposure_per_dollar"].isna().sum()
)

print(
    "Highest exposure per dollar:",
    round(exposure_ranking["exposure_per_dollar"].max(), 4)
)

print(
    "Lowest exposure per dollar:",
    round(exposure_ranking["exposure_per_dollar"].min(), 4)
)

display(
    exposure_ranking[
        [
            "item_id",
            "priority",
            "shortage_severity",
            "estimated_order_cost",
            "potential_lost_profit",
            "exposure_per_dollar"
        ]
    ].head(10).round(2)
)


Total products: 273
Missing exposure ratios: 0
Highest exposure per dollar: 0.4153
Lowest exposure per dollar: 0.0


,item_id,priority,shortage_severity,estimated_order_cost,potential_lost_profit,exposure_per_dollar
179,FOODS_1_207,MEDIUM,SEVERE,436.31,181.19,0.42
165,FOODS_3_300,MEDIUM,SEVERE,287.10,94.10,0.33
185,HOBBIES_1_337,MEDIUM,MODERATE,337.60,97.81,0.29
108,FOODS_3_148,MEDIUM,MODERATE,201.91,57.88,0.29
169,FOODS_2_311,MEDIUM,SEVERE,644.49,184.01,0.29
27,FOODS_3_377,HIGH,SEVERE,272.65,76.47,0.28
1,FOODS_3_252,HIGH,SEVERE,8474.55,2315.70,0.27
19,FOODS_1_200,HIGH,SEVERE,2924.96,766.72,0.26
147,FOODS_3_742,MEDIUM,SEVERE,513.00,133.13,0.26
24,FOODS_2_360,HIGH,SEVERE,2755.20,714.35,0.26



### 15. Compare Budget Allocation Strategies


In [15]:

# Create a reusable budget allocation function
def allocate_budget(data, budget):
    remaining = round(float(budget), 2)
    allocated = []

    for _, row in data.iterrows():
        unit_cost = float(row["supplier_unit_cost"])
        requested = int(row["recommended_order_qty"])

        affordable = int(
            (remaining + 1e-9) // unit_cost
        )

        quantity = min(requested, max(affordable, 0))
        allocated.append(quantity)

        remaining = round(
            remaining - quantity * unit_cost, 2
        )

    result = data.copy()
    result["new_allocated_qty"] = allocated

    result["new_allocated_cost"] = (
        result["new_allocated_qty"]
        * result["supplier_unit_cost"]
    ).round(2)

    return result

# Strategy A: Highest total financial exposure first
strategy_a = allocate_budget(
    recommendations.sort_values(
        "potential_lost_profit",
        ascending=False
    ),
    purchasing_budget
)

# Strategy B: Highest exposure per dollar first
strategy_b = allocate_budget(
    exposure_ranking,
    purchasing_budget
)

# Compare both strategies
comparison = pd.DataFrame({
    "Strategy": [
        "Total Financial Exposure",
        "Exposure per Dollar"
    ],
    "Funded Products": [
        (strategy_a["new_allocated_qty"] > 0).sum(),
        (strategy_b["new_allocated_qty"] > 0).sum()
    ],
    "Allocated Units": [
        strategy_a["new_allocated_qty"].sum(),
        strategy_b["new_allocated_qty"].sum()
    ],
    "Total Cost": [
        strategy_a["new_allocated_cost"].sum(),
        strategy_b["new_allocated_cost"].sum()
    ]
})

print(comparison.round(2))

assert (comparison["Total Cost"] <= purchasing_budget + 0.01).all()
print("\nBoth strategies respect the budget: PASSED")


                   Strategy  Funded Products  Allocated Units  Total Cost
0  Total Financial Exposure               19             5370    99999.72
1       Exposure per Dollar               56             7530    99999.80

Both strategies respect the budget: PASSED



### 16. Compare Unfunded Stockout Risk Across Strategies


In [17]:

# Copy shortage severity from budget_plan
severity_lookup = budget_plan.set_index(
    "item_id"
)["shortage_severity"]

# Add missing information to both strategies
strategy_a["shortage_severity"] = (
    strategy_a["item_id"].map(severity_lookup)
)

strategy_b["shortage_severity"] = (
    strategy_b["item_id"].map(severity_lookup)
)

# Verify the fix
print(
    "Strategy A missing severity:",
    strategy_a["shortage_severity"].isna().sum()
)

print(
    "Strategy B missing severity:",
    strategy_b["shortage_severity"].isna().sum()
)

print("Missing column fixed!")


Strategy A missing severity: 0
Strategy B missing severity: 0
Missing column fixed!



### 17. Compare Unfunded Stockout Risk Across Strategies


In [19]:

def summarize_allocation_risk(data, strategy_name):

    at_risk = data["stockout_risk"] == "AT_RISK"
    unfunded = data["new_allocated_qty"] == 0

    return {
        "Strategy": strategy_name,

        "Funded Products": (
            data["new_allocated_qty"] > 0
        ).sum(),

        "Unfunded At-Risk Products": (
            at_risk & unfunded
        ).sum(),

        "Unfunded Severe Products": (
            (data["shortage_severity"] == "SEVERE")
            & unfunded
        ).sum(),

        "Unfunded Requested Units": (
            data["recommended_order_qty"]
            - data["new_allocated_qty"]
        ).sum()
    }


risk_comparison = pd.DataFrame([
    summarize_allocation_risk(
        strategy_a, "Total Financial Exposure"
    ),
    summarize_allocation_risk(
        strategy_b, "Exposure per Dollar"
    )
])

display(risk_comparison)


,Strategy,Funded Products,Unfunded At-Risk Products,Unfunded Severe Products,Unfunded Requested Units
0,Total Financial Exposure,19,154,23,13675
1,Exposure per Dollar,56,117,4,11515



### 18. Compare Budget Coverage by Inventory Priority


In [20]:

# Count funded products by inventory priority
def priority_coverage(data, strategy_name):

    result = data.groupby("priority").agg(
        total_products=("item_id", "count"),
        funded_products=(
            "new_allocated_qty",
            lambda x: (x > 0).sum()
        )
    ).reset_index()

    result["coverage_percent"] = (
        result["funded_products"]
        / result["total_products"]
        * 100
    )

    result["strategy"] = strategy_name

    return result


# Compare both strategies
coverage_a = priority_coverage(
    strategy_a, "Strategy A"
)

coverage_b = priority_coverage(
    strategy_b, "Strategy B"
)

priority_comparison = pd.concat(
    [coverage_a, coverage_b],
    ignore_index=True
)

display(
    priority_comparison[
        [
            "strategy",
            "priority",
            "total_products",
            "funded_products",
            "coverage_percent"
        ]
    ].round(2)
)


,strategy,priority,total_products,funded_products,coverage_percent
0,Strategy A,HIGH,62,19,30.65
1,Strategy A,MEDIUM,211,0,0.00
2,Strategy B,HIGH,62,33,53.23
3,Strategy B,MEDIUM,211,23,10.90



### 19. Compare Reorder Quantity Coverage by Priority


In [21]:

def quantity_coverage(data, strategy_name):

    result = data.groupby("priority").agg(
        requested_units=("recommended_order_qty", "sum"),
        allocated_units=("new_allocated_qty", "sum")
    ).reset_index()

    result["quantity_coverage_percent"] = (
        result["allocated_units"]
        / result["requested_units"]
        * 100
    )

    result["strategy"] = strategy_name

    return result


quantity_a = quantity_coverage(
    strategy_a, "Strategy A"
)

quantity_b = quantity_coverage(
    strategy_b, "Strategy B"
)

quantity_comparison = pd.concat(
    [quantity_a, quantity_b],
    ignore_index=True
)

display(
    quantity_comparison[
        [
            "strategy",
            "priority",
            "requested_units",
            "allocated_units",
            "quantity_coverage_percent"
        ]
    ].round(2)
)


,strategy,priority,requested_units,allocated_units,quantity_coverage_percent
0,Strategy A,HIGH,10222,5370,52.53
1,Strategy A,MEDIUM,8823,0,0.00
2,Strategy B,HIGH,10222,6069,59.37
3,Strategy B,MEDIUM,8823,1461,16.56



### 20. Evaluate Delivery Timing Risk in Budget Allocations


In [23]:

# Get shortage information from our earlier analysis
gap_lookup = budget_plan.set_index(
    "item_id"
)["shortage_gap_days"]

# Add the missing column to both strategies
strategy_a["shortage_gap_days"] = (
    strategy_a["item_id"].map(gap_lookup)
)

strategy_b["shortage_gap_days"] = (
    strategy_b["item_id"].map(gap_lookup)
)

# Verify that nothing is missing
assert strategy_a["shortage_gap_days"].notna().all()
assert strategy_b["shortage_gap_days"].notna().all()


# Compare delivery risks
def delivery_risk_summary(data, strategy_name):

    funded = data["new_allocated_qty"] > 0
    at_risk = data["shortage_gap_days"] > 0
    severe = data["shortage_severity"] == "SEVERE"

    return {
        "Strategy": strategy_name,
        "Funded Products": funded.sum(),
        "Funded but At Risk Before Delivery": (
            funded & at_risk
        ).sum(),
        "Funded but Severe Risk": (
            funded & severe
        ).sum()
    }


delivery_comparison = pd.DataFrame([
    delivery_risk_summary(strategy_a, "Strategy A"),
    delivery_risk_summary(strategy_b, "Strategy B")
])

display(delivery_comparison)


,Strategy,Funded Products,Funded but At Risk Before Delivery,Funded but Severe Risk
0,Strategy A,19,19,15
1,Strategy B,56,56,34



### 21. Generate Delivery-Aware Management Recommendations


In [24]:

# Use Strategy B as our demonstration allocation
management_plan = strategy_b.copy()

# Recommend an action based on shortage severity
management_plan["management_action"] = np.select(
    [
        management_plan["shortage_severity"] == "SEVERE",
        management_plan["shortage_severity"] == "MODERATE"
    ],
    [
        "CHECK_EXPEDITED_DELIVERY",
        "REVIEW_DELIVERY_SCHEDULE"
    ],
    default="STANDARD_MONITORING"
)

# Keep all decisions pending manager review
management_plan["decision_status"] = "PENDING"

# Summarize actions
print("Management action distribution:")
print(management_plan["management_action"].value_counts())

print(
    "\nPending manager decisions:",
    (management_plan["decision_status"] == "PENDING").sum()
)

print(
    "\nMissing actions:",
    management_plan["management_action"].isna().sum()
)

display(
    management_plan[
        [
            "item_id",
            "priority",
            "shortage_severity",
            "new_allocated_qty",
            "management_action",
            "decision_status"
        ]
    ].head(10)
)


Management action distribution:
management_action
REVIEW_DELIVERY_SCHEDULE    135
STANDARD_MONITORING         100
CHECK_EXPEDITED_DELIVERY     38
Name: count, dtype: int64

Pending manager decisions: 273

Missing actions: 0


,item_id,priority,shortage_severity,new_allocated_qty,management_action,decision_status
179,FOODS_1_207,MEDIUM,SEVERE,23,CHECK_EXPEDITED_DELIVERY,PENDING
165,FOODS_3_300,MEDIUM,SEVERE,29,CHECK_EXPEDITED_DELIVERY,PENDING
185,HOBBIES_1_337,MEDIUM,MODERATE,20,REVIEW_DELIVERY_SCHEDULE,PENDING
108,FOODS_3_148,MEDIUM,MODERATE,61,REVIEW_DELIVERY_SCHEDULE,PENDING
169,FOODS_2_311,MEDIUM,SEVERE,27,CHECK_EXPEDITED_DELIVERY,PENDING
27,FOODS_3_377,HIGH,SEVERE,133,CHECK_EXPEDITED_DELIVERY,PENDING
1,FOODS_3_252,HIGH,SEVERE,735,CHECK_EXPEDITED_DELIVERY,PENDING
19,FOODS_1_200,HIGH,SEVERE,181,CHECK_EXPEDITED_DELIVERY,PENDING
147,FOODS_3_742,MEDIUM,SEVERE,38,CHECK_EXPEDITED_DELIVERY,PENDING
24,FOODS_2_360,HIGH,SEVERE,140,CHECK_EXPEDITED_DELIVERY,PENDING



### 22. Export Business Decision and Risk Analysis Results


In [25]:

# Create output folder
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Define output files
decision_path = output_dir / "business_decision_plan.csv"
comparison_path = output_dir / "budget_strategy_comparison.csv"

# Save business decision plan
management_plan.to_csv(
    decision_path,
    index=False
)

# Save both budget strategies for comparison
comparison.to_csv(
    comparison_path,
    index=False
)

# Load saved files for verification
saved_decisions = pd.read_csv(decision_path)
saved_comparison = pd.read_csv(comparison_path)

print("Saved business decisions:", len(saved_decisions))
print("Saved budget strategies:", len(saved_comparison))

print(
    "Pending decisions:",
    (saved_decisions["decision_status"] == "PENDING").sum()
)

print(
    "Missing decision values:",
    saved_decisions.isna().sum().sum()
)

print(
    "Files saved successfully:",
    decision_path.exists() and comparison_path.exists()
)


Saved business decisions: 273
Saved budget strategies: 2
Pending decisions: 273
Missing decision values: 0
Files saved successfully: True



## Conclusion

Developed a business decision and risk analysis system using simulated supplier costs, inventory data, and a $100,000 purchasing budget.

### Key Results
- **Total estimated purchasing cost:** $258,329.08
- **Available budget:** $100,000
- **Products at risk of shortage:** 173
- **Potential shortage:** 7,286.45 units
- **Estimated potential lost gross profit:** $43,267.50

### Budget Strategy Comparison
- **Strategy A:** Funded 19 products (5,370 units).
- **Strategy B:** Funded 56 products (7,530 units).
- Strategy B provided broader coverage of high-priority products under the same budget.

### Business Decision System
- Identified 38 products requiring expedited-delivery review.
- Generated management actions for 273 products.
- Preserved all purchasing decisions as PENDING.
- Exported business decision and budget comparison files.

**Limitations:** Financial figures and supplier parameters are simulated. The allocation strategies do not guarantee prevention of stockouts or optimal purchasing decisions.

**Next:** Notebook 06 — Agentic AI Workflow and Tool Integration.
